# Simple Stage 3 Evaluation

This notebook evaluates the part of Stage 3 that can be checked with SHARD annotations. SHARD `annotation.csv` contains shelf-row coordinates, not product bounding boxes, so this notebook reports **row agreement** for localized product boxes plus simple diagnostics. It does not compute product detection AP, precision, or recall.

In [ ]:
from pathlib import Path
import csv
import json
import math
from collections import Counter, defaultdict

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists() and PROJECT_ROOT.name == 'scripts':
    PROJECT_ROOT = PROJECT_ROOT.parent

LOCALIZED_PATH = PROJECT_ROOT / 'data/processed/s3_localized_products.csv'
ANNOTATIONS_PATH = PROJECT_ROOT / 'data/processed/SHARD/annotation.csv'
SHARD_INDEX_PATH = PROJECT_ROOT / 'data/processed/SHARD/shard_index.csv'
OUTPUT_DIR = PROJECT_ROOT / 'result/stage3_simple_eval'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LOCALIZED_PATH, ANNOTATIONS_PATH, SHARD_INDEX_PATH, OUTPUT_DIR

In [ ]:
def as_bool(value):
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in {'true', '1', 'yes', 'y'}

def to_float(value, default=None):
    if value in (None, ''):
        return default
    try:
        return float(value)
    except (TypeError, ValueError):
        return default

def to_int(value, default=None):
    if value in (None, ''):
        return default
    try:
        return int(float(value))
    except (TypeError, ValueError):
        return default

def mean(values):
    return sum(values) / len(values) if values else None

def median(values):
    if not values:
        return None
    ordered = sorted(values)
    mid = len(ordered) // 2
    return ordered[mid] if len(ordered) % 2 else (ordered[mid - 1] + ordered[mid]) / 2

def percentile(values, pct):
    if not values:
        return None
    ordered = sorted(values)
    pos = (len(ordered) - 1) * pct
    lo = int(math.floor(pos))
    hi = int(math.ceil(pos))
    if lo == hi:
        return ordered[lo]
    weight = pos - lo
    return ordered[lo] * (1 - weight) + ordered[hi] * weight

def safe_div(num, den):
    return num / den if den else None

In [ ]:
def read_localized(path):
    if path.suffix.lower() == '.json':
        rows = json.loads(path.read_text(encoding='utf-8'))
    else:
        with path.open(newline='', encoding='utf-8') as f:
            rows = list(csv.DictReader(f))
    normalized = []
    for row in rows:
        item = dict(row)
        for key in ('x1', 'y1', 'x2', 'y2', 'score'):
            if key in item:
                item[key] = to_float(item[key])
        for key in ('shelf_row', 'column', 'subrow'):
            if key in item:
                item[key] = to_int(item[key])
        item['discarded'] = as_bool(item.get('discarded', False))
        normalized.append(item)
    return normalized

def read_shard_annotations(path):
    rows_by_image = {}
    with path.open(newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f, delimiter=';')
        for row in reader:
            image = row.get('filename') or row.get('image') or row.get('image_name')
            coords = row.get('shelfCoord') or row.get('rows_normalized')
            if not image or not coords:
                continue
            values = [float(v.strip()) for v in coords.replace(';', ',').split(',') if v.strip()]
            rows_by_image[image] = sorted(values)
    return rows_by_image

def read_image_heights(path):
    if path is None or not path.exists():
        return {}
    heights = {}
    with path.open(newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            image = row.get('image') or row.get('filename') or row.get('image_name')
            height = to_float(row.get('height') or row.get('image_height'))
            if image and height:
                heights[image] = height
    return heights

def expected_row_from_gt(center_y_frac, gt_rows):
    assigned = len(gt_rows) + 1
    for idx, row_y in enumerate(gt_rows):
        if center_y_frac < row_y:
            assigned = idx + 1
            break
    return assigned

In [ ]:
localized = read_localized(LOCALIZED_PATH)
annotations = read_shard_annotations(ANNOTATIONS_PATH)
image_heights = read_image_heights(SHARD_INDEX_PATH)

len(localized), len(annotations), len(image_heights)

In [ ]:
by_image = defaultdict(list)
for item in localized:
    image = item.get('image_name') or item.get('image') or item.get('filename')
    if image:
        by_image[str(image)].append(item)

kept = [item for item in localized if not item.get('discarded', False)]
discarded = [item for item in localized if item.get('discarded', False)]
scores = [float(item['score']) for item in localized if to_float(item.get('score')) is not None]
kept_scores = [float(item['score']) for item in kept if to_float(item.get('score')) is not None]
discarded_scores = [float(item['score']) for item in discarded if to_float(item.get('score')) is not None]

row_total = row_correct = row_off_by_one = 0
row_abs_errors = []
row_confusion = Counter()
missing_annotation_images = set()
per_image = []

for image, items in sorted(by_image.items()):
    gt_rows = annotations.get(image)
    image_kept = [item for item in items if not item.get('discarded', False)]
    image_total = image_correct = image_off_by_one = 0
    image_abs_errors = []
    if not gt_rows:
        missing_annotation_images.add(image)
    else:
        for item in image_kept:
            pred_row = to_int(item.get('shelf_row'))
            y1 = to_float(item.get('y1'))
            y2 = to_float(item.get('y2'))
            if pred_row is None or y1 is None or y2 is None:
                continue
            image_height = image_heights.get(image, 1000.0)
            center_y_frac = ((y1 + y2) / 2.0) / image_height
            gt_row = expected_row_from_gt(center_y_frac, gt_rows)
            error = abs(pred_row - gt_row)
            row_total += 1
            image_total += 1
            row_abs_errors.append(float(error))
            image_abs_errors.append(float(error))
            row_confusion[(gt_row, pred_row)] += 1
            if error == 0:
                row_correct += 1
                image_correct += 1
            if error <= 1:
                row_off_by_one += 1
                image_off_by_one += 1
    per_image.append({
        'image_name': image,
        'gt_rows_available': bool(gt_rows),
        'num_gt_shelf_lines': len(gt_rows or []),
        'num_boxes_total': len(items),
        'num_boxes_kept': len(image_kept),
        'num_boxes_discarded': len(items) - len(image_kept),
        'row_eval_boxes': image_total,
        'row_agreement': safe_div(image_correct, image_total),
        'row_off_by_one_or_exact_rate': safe_div(image_off_by_one, image_total),
        'mean_abs_row_error': mean(image_abs_errors),
    })

In [ ]:
products_per_image = [len([item for item in items if not item.get('discarded', False)]) for items in by_image.values()]
products_per_row = Counter((item.get('image_name'), item.get('shelf_row')) for item in kept if item.get('shelf_row') is not None)
row_distribution = Counter(str(item.get('shelf_row')) for item in kept if item.get('shelf_row') is not None)
subrow_distribution = Counter(str(item.get('subrow')) for item in kept if item.get('subrow') is not None)

summary = {
    'metric_name': 'GT shelf-row agreement for Stage 3 localized boxes',
    'important_limitation': 'SHARD annotation.csv does not contain product bounding boxes, so this does not measure product detection AP, precision, recall, or true product localization accuracy.',
    'num_images_in_predictions': len(by_image),
    'num_images_with_shard_annotations': len(set(by_image) & set(annotations)),
    'num_images_missing_shard_annotations': len(missing_annotation_images),
    'num_predictions_total': len(localized),
    'num_predictions_kept': len(kept),
    'num_predictions_discarded': len(discarded),
    'discard_rate': safe_div(len(discarded), len(localized)),
    'row_eval_boxes': row_total,
    'row_agreement': safe_div(row_correct, row_total),
    'row_off_by_one_or_exact_rate': safe_div(row_off_by_one, row_total),
    'mean_abs_row_error': mean(row_abs_errors),
    'median_abs_row_error': median(row_abs_errors),
    'avg_products_per_image': mean([float(v) for v in products_per_image]),
    'median_products_per_image': median([float(v) for v in products_per_image]),
    'avg_products_per_nonempty_row': mean([float(v) for v in products_per_row.values()]),
    'stacked_product_rate': safe_div(sum(1 for item in kept if to_int(item.get('subrow'), 1) and to_int(item.get('subrow'), 1) > 1), len(kept)),
    'score_mean': mean(scores),
    'score_median': median(scores),
    'score_p95': percentile(scores, 0.95),
    'kept_score_mean': mean(kept_scores),
    'discarded_score_mean': mean(discarded_scores),
    'row_distribution': dict(sorted(row_distribution.items())),
    'subrow_distribution': dict(sorted(subrow_distribution.items())),
    'row_confusion': [{'gt_row_from_shard': gt, 'stage3_row': pred, 'count': count} for (gt, pred), count in sorted(row_confusion.items())],
}

summary

In [ ]:
summary_path = OUTPUT_DIR / 'summary.json'
per_image_path = OUTPUT_DIR / 'per_image.csv'

summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
with per_image_path.open('w', newline='', encoding='utf-8') as f:
    fieldnames = list(per_image[0].keys()) if per_image else []
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(per_image)

summary_path, per_image_path